# Proyecto Final Big Data — Tokio Telecom
## Análisis de bajas (churn) de clientes con **PySpark** en Google Colab

**Dataset:** `customer_churn_10k.csv` (10.000 clientes, 12 columnas)

Este notebook resuelve las **tareas 3 a 6** del proyecto usando exclusivamente **Python + Spark (PySpark)**:

- **T3** — Carga en DataFrame + análisis de churn por tipo de tarifa y por duración de contrato.
- **T4** — Análisis descriptivo (media y desviación estándar).
- **T5** — Análisis de correlación (matriz de Pearson).
- **T6** — Aprendizaje supervisado: Random Forest + Regresión Logística (baseline) para predecir la baja.


## 0. Instalación de PySpark en Colab
Colab ya trae Java. Solo instalamos la librería `pyspark`.

In [ ]:
!pip install pyspark

## 1. Crear la sesión de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (SparkSession.builder
         .appName('Tokio_Telecom_Churn')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

## 2. Subir y cargar el CSV
Ejecuta la celda y selecciona `customer_churn_10k.csv` desde tu equipo.
(Alternativa: súbelo al panel de archivos de Colab y comenta la línea de `files.upload()`.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ruta = 'customer_churn_10k.csv'

### 2.1 Cargar el conjunto de datos en un DataFrame de PySpark

In [ ]:
df = spark.read.csv(ruta, header=True, inferSchema=True)
# CustomerID debe tratarse como CATEGÓRICA y no usarse en los modelos
df = df.withColumn('CustomerID', F.col('CustomerID').cast('string'))
print('Filas:', df.count(), '| Columnas:', len(df.columns))
df.printSchema()
df.show(5, truncate=False)

### 2.2 Estructura de los datos (Nombre del campo / Tipo)
Referencia de la Tarea 1 — tipos con los que trabajaremos:

In [ ]:
import pandas as pd
tipos = pd.DataFrame({
    'Nombre del Campo': [f.name for f in df.schema.fields],
    'Tipo de dato':     [f.dataType.simpleString() for f in df.schema.fields]
})
tipos

---
## Tarea 3 — Clientes y % de churn por categoría
Calculamos, para **Subscription Type** y **Contract Length**, cuántos clientes hay y qué porcentaje ha hecho churn (`Churn_YesNo = 'Yes'`).

In [ ]:
def churn_por(col):
    return (df.groupBy(col)
              .agg(F.count('*').alias('clientes'),
                   F.sum(F.when(F.col('Churn_YesNo')=='Yes',1).otherwise(0)).alias('churn_yes'))
              .withColumn('pct_churn', F.round(100*F.col('churn_yes')/F.col('clientes'),2))
              .orderBy(F.desc('pct_churn')))

t3_sub = churn_por('Subscription Type')
t3_con = churn_por('Contract Length')
t3_sub.show(truncate=False)
t3_con.show(truncate=False)

### Gráficos de la Tarea 3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
AZUL, ROSA = '#1a1aa8', '#f96b7d'

def grafico_churn(spark_df, catcol, titulo):
    pdf = spark_df.toPandas().sort_values('pct_churn', ascending=False).reset_index(drop=True)
    fig, ax1 = plt.subplots(figsize=(7,4.3))
    x = np.arange(len(pdf))
    ax1.bar(x-0.2, pdf['clientes'], 0.4, color=AZUL, label='Nº clientes')
    ax1.set_ylabel('Nº de clientes', color=AZUL); ax1.tick_params(axis='y', labelcolor=AZUL)
    ax2 = ax1.twinx()
    ax2.bar(x+0.2, pdf['pct_churn'], 0.4, color=ROSA, label='% churn')
    ax2.set_ylabel('% churn (Yes)', color=ROSA); ax2.tick_params(axis='y', labelcolor=ROSA)
    ax2.set_ylim(0, max(105, pdf['pct_churn'].max()*1.2))
    for i,v in enumerate(pdf['pct_churn']): ax2.text(i+0.2, v+1.5, f'{v:.1f}%', ha='center', color=ROSA, fontweight='bold', fontsize=9)
    ax1.set_xticks(x); ax1.set_xticklabels(pdf[catcol]); ax1.set_title(titulo, fontweight='bold')
    fig.tight_layout(); plt.show()

grafico_churn(t3_sub, 'Subscription Type', 'Churn por tipo de tarifa')
grafico_churn(t3_con, 'Contract Length', 'Churn por duración del contrato')

> **Interpretación de negocio (T3).** Por tarifa el churn es alto y homogéneo (Basic 58,3% · Standard 56,7% · Premium 55,3%): el tipo de tarifa apenas discrimina la baja. En cambio la **duración del contrato es determinante**: el contrato **mensual presenta un 100% de churn**, frente al ~47% del trimestral y ~45% del anual. El compromiso a medio/largo plazo es la palanca de retención más potente.

---
## Tarea 4 — Análisis descriptivo (media y desviación estándar)
Solo variables **numéricas** (excluimos `CustomerID`, que es categórica).

In [ ]:
num_cols = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction']
desc = df.select([F.round(F.mean(c),3).alias(c+'_media') for c in num_cols] +
                 [F.round(F.stddev(c),3).alias(c+'_desv') for c in num_cols])
# Presentación en formato largo y legible
filas = []
for c in num_cols:
    r = df.select(F.mean(c).alias('m'), F.stddev(c).alias('s'),
                  F.min(c).alias('mn'), F.max(c).alias('mx')).first()
    filas.append((c, round(r['m'],3), round(r['s'],3), r['mn'], r['mx']))
import pandas as pd
pd.DataFrame(filas, columns=['Variable','Media','Desv. Estándar','Mín','Máx'])

> **Interpretación (T4).** Edad media 39 años (18–65). `Tenure` (antigüedad) media 31 meses con alta dispersión (±17). `Support Calls` media 3,6 y `Payment Delay` media 13 días: variables de fricción con recorrido. `Total Spend` medio 633 € (±240), muy dispersa → conviven clientes de bajo y alto valor.

---
## Tarea 5 — Análisis de correlación
Para calcular la matriz de correlación necesitamos **ensamblar** las variables numéricas en un vector (`VectorAssembler`) y aplicar `Correlation.corr` (Pearson).

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

df_num = df.select([F.col(c).cast(DoubleType()).alias(c) for c in num_cols])
vec = VectorAssembler(inputCols=num_cols, outputCol='features_corr').transform(df_num).select('features_corr')
M = Correlation.corr(vec, 'features_corr', 'pearson').head()[0].toArray()
import pandas as pd
corr_df = pd.DataFrame(M, index=num_cols, columns=num_cols).round(3)
corr_df

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(figsize=(7.5,6.2))
im = ax.imshow(M, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels([c.replace(' ','\n') for c in num_cols], fontsize=8); ax.set_yticklabels(num_cols, fontsize=8)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j,i,f'{M[i,j]:.2f}', ha='center', va='center', color='white' if abs(M[i,j])>0.5 else 'black', fontsize=8)
ax.set_title('Matriz de correlación (Pearson)', fontweight='bold'); fig.colorbar(im, fraction=0.046, pad=0.04)
fig.tight_layout(); plt.show()

### Correlación de cada variable con la baja
Añadimos una columna numérica `churn01` (Yes=1) para ver qué variables se asocian más con el churn.

In [ ]:
df_c = df.withColumn('churn01', F.when(F.col('Churn_YesNo')=='Yes',1.0).otherwise(0.0))
for c in num_cols:
    print(f'{c:18s} corr con churn = {df_c.stat.corr(c, "churn01"):.3f}')

> **Interpretación (T5).** Entre las variables numéricas la correlación es muy baja (no hay multicolinealidad → todas aportan información propia a un modelo). Frente a la baja destacan: **Support Calls (+0,58)**, **Total Spend (−0,42)**, **Payment Delay (+0,32)** y **Age (+0,23)**. Lectura: más llamadas a soporte y más retrasos de pago empujan a la baja; más gasto (cliente más comprometido/valioso) la reduce.

---
## Tarea 6 — Aprendizaje supervisado (predecir la baja)
**Problema de negocio.** Anticipar qué clientes van a darse de baja permite actuar (retención) *antes* de perderlos, protegiendo ingresos recurrentes. Es un problema de **clasificación binaria** con etiqueta conocida (`Churn_YesNo`), por lo que usamos **aprendizaje supervisado**.

**Modelos.** Entrenamos un **Random Forest** (robusto y con *importancia de variables* → nos dice qué causa la baja) y una **Regresión Logística** como *baseline* comparativo.

**Procesado de categóricas.** `Gender`, `Subscription Type` y `Contract Length` se transforman con `StringIndexer` + `OneHotEncoder`. `CustomerID` se excluye.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Etiqueta: Yes = 1 (la baja es la clase positiva)
data = df.withColumn('label', F.when(F.col('Churn_YesNo')=='Yes',1.0).otherwise(0.0))

numeric_cols     = ['Age','Tenure','Usage Frequency','Support Calls','Payment Delay','Total Spend','Last Interaction']
categorical_cols = ['Gender','Subscription Type','Contract Length']

indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe') for c in categorical_cols]
assembler = VectorAssembler(inputCols=numeric_cols+[c+'_ohe' for c in categorical_cols], outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaledFeatures', withStd=True, withMean=False)
pre = indexers + encoders + [assembler, scaler]

### 6.1 Partición train/test y entrenamiento

In [ ]:
train, test = data.randomSplit([0.7, 0.3], seed=42)
print('Train:', train.count(), '| Test:', test.count())

rf = RandomForestClassifier(labelCol='label', featuresCol='features', numTrees=100, maxDepth=6, seed=42)
lr = LogisticRegression(labelCol='label', featuresCol='scaledFeatures', maxIter=50)

model_rf = Pipeline(stages=pre+[rf]).fit(train)
model_lr = Pipeline(stages=pre+[lr]).fit(train)
pred_rf = model_rf.transform(test)
pred_lr = model_lr.transform(test)

### 6.2 Evaluación y comparación

In [ ]:
auc = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
def evaluar(pred, nombre):
    acc = MulticlassClassificationEvaluator(labelCol='label', metricName='accuracy').evaluate(pred)
    f1  = MulticlassClassificationEvaluator(labelCol='label', metricName='f1').evaluate(pred)
    pr  = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedPrecision').evaluate(pred)
    rc  = MulticlassClassificationEvaluator(labelCol='label', metricName='weightedRecall').evaluate(pred)
    ar  = auc.evaluate(pred)
    print(f'[{nombre}] Accuracy={acc:.4f} Precision={pr:.4f} Recall={rc:.4f} F1={f1:.4f} AUC={ar:.4f}')
    return dict(nombre=nombre, acc=acc, pr=pr, rc=rc, f1=f1, auc=ar)

m_rf = evaluar(pred_rf, 'Random Forest')
m_lr = evaluar(pred_lr, 'Regresión Logística')

#### Matrices de confusión

In [ ]:
import numpy as np, matplotlib.pyplot as plt
def matriz(pred):
    M = np.zeros((2,2))
    for r in pred.groupBy('label','prediction').count().collect():
        M[int(r['label']), int(r['prediction'])] = r['count']
    return M
cms = [(matriz(pred_rf),'Random Forest'), (matriz(pred_lr),'Regresión Logística')]
fig, axes = plt.subplots(1,2, figsize=(10,4.3))
for ax,(M,nom) in zip(axes, cms):
    ax.imshow(M, cmap='Blues'); ax.set_title(nom, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['No','Yes']); ax.set_yticklabels(['No','Yes'])
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
    for i in range(2):
        for j in range(2):
            ax.text(j,i,int(M[i,j]), ha='center', va='center', color='white' if M[i,j]>M.max()/2 else 'black', fontsize=12, fontweight='bold')
fig.suptitle('Matrices de confusión (test)', fontweight='bold'); fig.tight_layout(); plt.show()

#### Comparación de métricas

In [ ]:
labels_m = ['Accuracy','Precision','Recall','F1','AUC']
rf_v = [m_rf['acc'],m_rf['pr'],m_rf['rc'],m_rf['f1'],m_rf['auc']]
lr_v = [m_lr['acc'],m_lr['pr'],m_lr['rc'],m_lr['f1'],m_lr['auc']]
x = np.arange(len(labels_m)); w=0.38
fig, ax = plt.subplots(figsize=(8,4.5))
b1=ax.bar(x-w/2, rf_v, w, color='#1a1aa8', label='Random Forest')
b2=ax.bar(x+w/2, lr_v, w, color='#f96b7d', label='Regresión Logística')
ax.set_ylim(0,1.08); ax.set_xticks(x); ax.set_xticklabels(labels_m); ax.set_ylabel('Valor')
ax.set_title('Random Forest vs Regresión Logística', fontweight='bold')
for bs in (b1,b2):
    for b in bs: ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{b.get_height():.3f}', ha='center', fontsize=8)
ax.legend(); fig.tight_layout(); plt.show()

### 6.3 Importancia de variables (Random Forest)

In [ ]:
feat_names = list(numeric_cols)
idx_models = {s.getInputCol(): s for s in model_rf.stages if hasattr(s,'labels')}
for c in categorical_cols:
    for lab in idx_models[c].labels[:-1]:  # OHE dropLast elimina la última categoría
        feat_names.append(f'{c}={lab}')
imp = model_rf.stages[-1].featureImportances.toArray()
pares = sorted(zip(feat_names, imp), key=lambda x:x[1], reverse=True)
for n,v in pares: print(f'{n:28s} {v:.4f}')

top = pares[:10][::-1]
fig, ax = plt.subplots(figsize=(8,5))
ax.barh([t[0] for t in top], [t[1] for t in top], color='#5bc4a8')
ax.set_title('Importancia de variables — Random Forest', fontweight='bold'); ax.set_xlabel('Importancia relativa')
for i,t in enumerate(top): ax.text(t[1]+0.003, i, f'{t[1]:.3f}', va='center', fontsize=8)
fig.tight_layout(); plt.show()

> **Conclusiones del modelo (T6).**
> - El **Random Forest** clasifica casi perfectamente la baja: **Accuracy 98,4%**, **AUC 0,996**, F1 0,985; muy por encima de la **Regresión Logística** (Accuracy 89,5%, AUC 0,958), que aun así es un baseline sólido.
> - Las variables que **más pesan** en la predicción son **Support Calls**, **Total Spend**, **Age** y **Payment Delay** — coherente con la Tarea 5.
> - **Acción de negocio:** vigilar clientes con muchas llamadas a soporte y retrasos de pago, y diseñar incentivos para migrar del contrato mensual (100% de baja) a trimestral/anual.

In [ ]:
spark.stop()